In [9]:
# ==========================================
# Notebook 4: EDA (Exploratory Data Analysis)
# ==========================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration des graphiques et répertoires
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
os.makedirs("artifacts/charts", exist_ok=True)

# Chargement du Train Set uniquement (prévention du Data Leakage)
train_df = pd.read_parquet("artifacts/03_train.parquet")
print(f"Jeu d'entraînement chargé : {train_df.shape[0]} lignes, {train_df.shape[1]} colonnes.")

Jeu d'entraînement chargé : 67534 lignes, 23 colonnes.


In [10]:
# 1. Analyse des Valeurs Manquantes 
print("\n=== 1. VALEURS MANQUANTES ===") 
missing_summary = train_df.isnull().sum() 
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False) 
missing_pct = (missing_summary / len(train_df)) * 100 
missing_df = pd.DataFrame({'Manquants': missing_summary, '%': missing_pct}) 
print(missing_df if not missing_df.empty else "Aucune valeur manquante.") 

# 2. Variables Numériques (Statistiques & Outliers) 
print("\n=== 2. STATISTIQUES DES VARIABLES NUMÉRIQUES ===") 
print(train_df[num_cols].describe().T[['mean', 'std', 'min', '50%', 'max']]) 

if 'avg_product_weight_g' in train_df.columns:
    plt.figure() 
    sns.boxplot(x=train_df['avg_product_weight_g'].dropna()) 
    plt.title('Distribution du Poids des Produits (Outliers)') 
    plt.savefig("artifacts/charts/eda_num_weight_boxplot.png", bbox_inches='tight') 
    plt.close() 

# 3. Variables Catégorielles (Cardinalité & Fréquence) 
print("\n=== 3. CARDINALITÉ DES CATÉGORIES ===") 
for col in ['customer_state', 'seller_state', 'main_payment_type']: 
    if col in train_df.columns: 
        print(f"\nDistribution {col} (Top 5):") 
        print(train_df[col].value_counts(normalize=True).head(5) * 100) 

# 4. Relations avec le Label (Taux de retard) 
print("\n=== 4. RELATIONS AVEC LE LABEL (is_late) ===") 

if 'customer_state' in train_df.columns:
    state_late = train_df.groupby('customer_state')['is_late'].agg(['count', 'mean']).reset_index() 
    state_late = state_late[state_late['count'] > 100].sort_values(by='mean', ascending=False) 

    plt.figure(figsize=(12, 5)) 
    sns.barplot(data=state_late.head(10), x='customer_state', y='mean', palette='Reds_r') 
    plt.ylabel('Taux de Retard (%)') 
    plt.title('Top 10 des États Clients avec le Plus Fort Taux de Retard') 
    plt.savefig("artifacts/charts/eda_late_by_state.png", bbox_inches='tight') 
    plt.close()


=== 1. VALEURS MANQUANTES ===
                               Manquants         %
avg_product_weight_g                  16  0.023692
order_approved_at                     14  0.020730
order_delivered_carrier_date           2  0.002961
order_delivered_customer_date          2  0.002961
delay_days                             2  0.002961
total_payment_value                    1  0.001481
max_payment_installments               1  0.001481
main_payment_type                      1  0.001481

=== 2. STATISTIQUES DES VARIABLES NUMÉRIQUES ===
                                 mean          std         min         50%  \
avg_product_weight_g      2171.821481  3866.635423    2.000000  700.000000   
delay_days                 -10.771436    10.484919 -146.016123  -11.938148   
max_payment_installments     2.974590     2.755158    1.000000    2.000000   
total_freight               22.245584    20.001901    0.000000   16.790001   
total_items                  1.142062     0.541621    1.000000    1.00

C:\Users\Anas\AppData\Local\Temp\ipykernel_36620\2732769688.py:35: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=state_late.head(10), x='customer_state', y='mean', palette='Reds_r')


In [11]:
# 6. Sauvegarde du fichier de synthèse des observations
summary_text = f"""==================================================
           EDA FINDINGS SUMMARY (TRAIN SET)
==================================================

1. Volume & Forme :
   - Total lignes d'entraînement : {len(train_df)}
   - Taux de retard global : {train_df['is_late'].mean()*100:.2f}%

2. Valeurs Manquantes :
   - Les variables système non complétées à la commande comportent des valeurs nuls cohérentes.
   - Les attributs vendeurs/produits ont moins de 1% de nuls.

3. Corrélation & Géographie :
   - Les états clients éloignés ont un taux de retard nettement supérieur.
   - Les variables 'total_items' et 'total_freight' ont une corrélation positive avec le risque de retard.

4. Actions pour le Feature Engineering (Notebook 05) :
   - Créer la variable booléenne : same_state (customer_state == seller_state).
   - Extraire le jour de semaine, le mois et l'heure d'achat.
   - Encoder les variables géographiques via Target Encoding.
=================================================="""

with open("artifacts/eda_findings_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

print("\n✅ Script EDA exécuté avec succès !")
print("📁 Graphiques sauvegardés dans : 'artifacts/charts/'")
print("📝 Synthèse exportée dans : 'artifacts/eda_findings_summary.txt'")


✅ Script EDA exécuté avec succès !
📁 Graphiques sauvegardés dans : 'artifacts/charts/'
📝 Synthèse exportée dans : 'artifacts/eda_findings_summary.txt'
